# 🛡️ BeltGuard — Training (LOKAL versiya, Windows)

Bu notebook `g:\belt_damage` papkasida lokal ishlaydi. GPU: **GTX 1650 (4 GB)** — CUDA torch o'rnatilgan.

> 💡 4 GB VRAM uchun sozlangan: `yolo11s-seg`, `batch=8`, `imgsz=640`.
> Agar **CUDA out of memory** xatosi chiqsa: `batch=4` qiling, baribir chiqsa `yolo11n-seg` ga o'ting.
> Training oldidan og'ir dasturlarni (Chrome ko'p tab, o'yin) yoping — VRAM bo'shaydi.

Bosqichlar: muhit → (video→kadrlar, ixtiyoriy) → Roboflow dataset → demo-video → training → baholash → model `models/` ga

## 1. Muhit (streamlit/ultralytics allaqachon o'rnatilgan — faqat roboflow qo'shiladi)

In [ ]:
%pip install -q roboflow

import ultralytics
ultralytics.checks()  # CPU chiqadi — bu normal

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()          # g:\belt_damage
VIDEO_DIR   = PROJECT_DIR / 'videos'
FRAMES_DIR  = PROJECT_DIR / 'frames'
DS_DIR      = PROJECT_DIR / 'ds_public'
MODELS_DIR  = PROJECT_DIR / 'models'
for d in (VIDEO_DIR, FRAMES_DIR, MODELS_DIR):
    d.mkdir(exist_ok=True)
print('Loyiha papkasi:', PROJECT_DIR)

## 2. Video → kadrlar (IXTIYORIY — video bo'lmasa tashlab keting)

Bu bosqich faqat o'z stendingizda video yozgan bo'lsangiz kerak. **Video yo'q bo'lsa to'g'ridan-to'g'ri 3-bosqichga o'ting** — tayyor dataset bilan hammasi ishlaydi.

> ⚠️ **Test uchun ajratgan videoni bu yerga qo'shmang!** Uni faqat 6-bosqichda ishlatasiz.

In [ ]:
import cv2

FRAME_STEP = 7        # har 7-kadr (30fps da ~4 kadr/sekund)
MAX_BLUR = 60.0       # xira kadrlarni tashlash chegarasi (Laplacian var)

saved, skipped_blur = 0, 0
for vid_path in sorted(VIDEO_DIR.iterdir()):
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened():
        continue
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % FRAME_STEP == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            blur = cv2.Laplacian(gray, cv2.CV_64F).var()
            if blur < MAX_BLUR:
                skipped_blur += 1
            else:
                cv2.imwrite(str(FRAMES_DIR / f'{vid_path.stem}_{i:06d}.jpg'), frame,
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
                saved += 1
        i += 1
    cap.release()
    print(f'{vid_path.name}: qayta ishlandi')

print(f'\nJami saqlandi: {saved} kadr | Xiralik sabab tashlandi: {skipped_blur}')
print('Kadrlarni Roboflow ga yuklab, SAM (Smart Polygon) bilan annotatsiya qiling:')
print('https://app.roboflow.com -> Create Project (Instance Segmentation) -> Upload')

## 3. Tayyor ochiq datasetni yuklab olish (Roboflow)

`conveyor-belt-damage` — 922 rasm, 9 klass, CC BY 4.0.
API kalit: app.roboflow.com → Settings → API Key (bepul akkaunt yetadi).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'N3vI6T7j5CPoQfE9ea8g'   # <-- almashtiring!

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('sample-wy2mp').project('conveyor-belt-damage')
dataset = project.version(1).download('yolov8', location=str(DS_DIR))
print('Yuklandi:', dataset.location)

In [ ]:
# data.yaml — klasslar ro'yxatini ko'ramiz (dashboard CLASS_SEVERITY ga moslash uchun)
print((DS_DIR / 'data.yaml').read_text())

## 4. Training (GTX 1650 uchun sozlangan)

`yolo11s-seg` + 640px + batch 8. GTX 1650 da ~2–3 soat chamasi.

**CUDA out of memory** chiqsa: avval `batch=4`, keyin ham chiqsa `yolo11n-seg`.

In [ ]:
from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), 'CUDA topilmadi — torch CPU versiyada. Chat orqali tuzatamiz.'
print('GPU:', torch.cuda.get_device_name(0))

DATA_YAML = str(DS_DIR / 'data.yaml')

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=8,              # OOM chiqsa 4 ga tushiring
    device=0,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    # Lenta uchun sozlangan augmentatsiya:
    mosaic=0.5,
    copy_paste=0.3,
    degrees=5,
    flipud=0.0,
    fliplr=0.5,
    hsv_v=0.5,
    hsv_s=0.4,
    translate=0.2,
    scale=0.5,
    erasing=0.3,
    project=str(PROJECT_DIR / 'runs'),
    name='v1_local',
)

## 4. Training

CPU uchun sozlangan: `yolo11n-seg` (nano) + 640px + batch 8. Baribir sekin bo'ladi —
vaqt yetmasa Colab'da `yolo11s-seg` bilan qiling, bu asosiy tavsiya.

In [ ]:
from ultralytics import YOLO

DATA_YAML = str(DS_DIR / 'data.yaml')

model = YOLO('yolo11n-seg.pt')   # CPU uchun nano; Colab'da 's' ishlating

results = model.train(
    data=DATA_YAML,
    epochs=40,            # CPU uchun qisqartirilgan; GPU bo'lsa 60-150
    imgsz=640,
    batch=8,
    patience=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    # Lenta uchun sozlangan augmentatsiya:
    mosaic=0.5,
    copy_paste=0.3,
    degrees=5,
    flipud=0.0,
    fliplr=0.5,
    hsv_v=0.5,
    hsv_s=0.4,
    translate=0.2,
    scale=0.5,
    erasing=0.3,
    project=str(PROJECT_DIR / 'runs'),
    name='v1_local',
)

## 5. Baholash — metrikalar

In [ ]:
BEST = PROJECT_DIR / 'runs' / 'v1_local' / 'weights' / 'best.pt'
model = YOLO(str(BEST))

metrics = model.val(data=DATA_YAML)
print(f'Segment mAP@50    : {metrics.seg.map50:.3f}')
print(f'Segment mAP@50-95 : {metrics.seg.map:.3f}')
print('\nHar klass bo`yicha mAP@50-95 (segment):')
for idx, cls_name in metrics.names.items():
    if idx < len(metrics.seg.maps):
        print(f'  {cls_name:20s}: {metrics.seg.maps[idx]:.3f}')

In [ ]:
# Confusion matrix va PR-curve rasmlari
from IPython.display import Image, display
for p in sorted((PROJECT_DIR / 'runs' / 'v1_local').glob('*.png'))[:6]:
    print(p.name)
    display(Image(str(p), width=700))

## 6. Test videoda vizual tekshiruv (training da ishlatilmagan video!)

In [ ]:
TEST_VIDEO = PROJECT_DIR / 'test_video.mp4'   # <-- test videongizni shu nom bilan qo'ying

if TEST_VIDEO.exists():
    model.predict(str(TEST_VIDEO), save=True, conf=0.35, imgsz=640,
                  project=str(PROJECT_DIR / 'preds'), name='test')
    print('Natija video:', PROJECT_DIR / 'preds' / 'test')
else:
    print('Test video topilmadi —', TEST_VIDEO, 'ga qo`ying')

## 7. Modelni dashboard'ga ulash

`files.download` kerak emas — model to'g'ridan-to'g'ri `models\best.pt` ga ko'chiriladi.

**Colab'da o'rgatgan bo'lsangiz:** u yerdan yuklab olgan `best.pt` ni qo'lda `models\best.pt` ga qo'ying — bu katak shart emas.

In [ ]:
import shutil

shutil.copy(BEST, MODELS_DIR / 'best.pt')
print('Tayyor:', MODELS_DIR / 'best.pt')
print('Endi dashboard ishlaydi:  streamlit run beltguard_dashboard.py')